In [3]:
import networkx as nx
import matplotlib.pyplot as plt
import time
import random
import os
import csv


def generate_subsets(vertices):
    subsets = []

    def make_subsets(index, current):
        if index == len(vertices):
            subsets.append(current[:])
            return

        current.append(vertices[index])
        make_subsets(index + 1, current)
        current.pop()

        make_subsets(index + 1, current)

    make_subsets(0, [])
    return subsets


def is_vertex_cover(edges, subset):
    for u, v in edges:
        if u not in subset and v not in subset:
            return False

    return True


def vertex_cover(vertices, edges):
    subsets = generate_subsets(vertices)
    best = vertices
    for subset in subsets:
        if len(subset) < len(best):
            if is_vertex_cover(edges, subset):
                best = subset

    return best


def random_graph(n, m):
    edges = set()
    vertices = list(range(1, n + 1))
    random.shuffle(vertices)
    for i in range(1, n):
        u = vertices[i]
        v = random.choice(vertices[:i])
        edges.add((min(u, v), max(u, v)))

    while len(edges) < m:
        u = random.randint(1, n)
        v = random.randint(1, n)
        if u != v:
            edge = (min(u, v), max(u, v))
            edges.add(edge)

    return list(edges)


def draw_graph(n, m, edges, cover):
    graph = nx.Graph()
    graph.add_edges_from(edges)
    colors = []
    for node in graph.nodes():
        if node in cover:
            colors.append("red")
        else:
            colors.append("blue")

    plt.figure(figsize=(8, 8))
    nx.draw(
        graph,
        with_labels=True,
        node_color=colors,
        node_size=1000,
        font_color="white"
    )

    plt.title(f"Minimum Vertex Cover\nn = {n}, m = {m}\nCover = {cover}")
    filename = f"all_outputs/images/graph_n{n}_m{m}.png"
    plt.savefig(filename, dpi=300)
    plt.close()


os.makedirs("all_inputs", exist_ok=True)
os.makedirs("all_outputs/images", exist_ok=True)

n = 10
m_values = [10, 15, 20, 25, 30, 35, 40, 45]

random.seed(42)

for i in range(len(m_values)):
    m = m_values[i]
    filename = f"all_inputs/input{i + 1}.txt"
    if not os.path.exists(filename):
        edges = random_graph(n, m)
        with open(filename, "w") as file:
            file.write(f"{n} {m}\n")
            for u, v in edges:
                file.write(f"{u} {v}\n")


with open("all_outputs/results.csv", "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "(n,m)",
        "BF Size",
        "BF Time"
    ])

    for i in range(1, 9):
        filename = f"all_inputs/input{i}.txt"
        with open(filename, "r") as file:
            n, m = map(int, file.readline().split())
            vertices = list(range(1, n + 1))
            edges = []

            for line in file:
                u, v = map(int, line.split())
                edges.append((u, v))

        start = time.perf_counter()
        cover = vertex_cover(vertices, edges)
        end = time.perf_counter()
        time_taken = end - start
        writer.writerow([
            f"({n},{m})",
            len(cover),
            f"{time_taken:.10f}"
        ])

        output = f"all_outputs/output{i}.txt"
        with open(output, "w") as file:
            file.write(f"(n,m) = ({n},{m})\n")
            file.write(f"Vertex Cover Size = {len(cover)}\n")
            file.write(f"Vertex Cover Nodes = {cover}\n")
            file.write(f"Running Time = {time_taken:.10f} seconds\n")

        draw_graph(n, m, edges, cover)

        # print(f"Input File = input{i}.txt")
        # print(f"n = {n}, m = {m}")
        # print(f"Vertex Cover = {cover}")
        # print(f"Size = {len(cover)}")
        # print(f"Running Time = {time_taken:.10f} seconds")
        # print("-" * 40)


print("\nPractical01 results saved.")


Practical01 results saved.
